<div style="text-align: justify; max-width: 90ch">

# Lecture 01a: Your laptop as an LLM server

**Status: Mandatory reading.**

In the Python course you trained models yourself: `fit`, `predict`, a metric. A **large language model (LLM)** is different:
someone else trained it for months on thousands of GPUs, and your job is to *run* it and *talk* to it. This notebook makes that concrete on your own laptop, with no account and no API key.

**What this notebook covers**

- Two new tools: `uv` for the course environment, and **Ollama**, a local model server.
- Servers, ports and clients; what running locally gains and costs.
- Your first call from Python: what you send, what the reply contains.
- **Tokens**: the unit a model reads, writes, and is paid in.
- The **context window**: how much text a model can see at once.
- **Temperature**: why the same question gives different answers.

</div>

<div style="text-align: justify; max-width: 90ch">

Before you start, finish the quick starts of the guides `01a_git_uv` to `01c_ollama` in this lecture's `infra_tools/` folder,
which we run together in class: `uv sync`, Ollama installed, `ollama pull qwen3:1.7b`.

</div>

In [1]:
import ollama
import pandas as pd
from transformers import AutoTokenizer

<div style="text-align: justify; max-width: 90ch">

## 0. The configuration cell

Every notebook in this course starts with the same cell. You edit **one line**: `TIER`, which picks the chat model.

- **`cpu`** uses [`qwen3:1.7b`](https://ollama.com/library/qwen3), a model small enough for any laptop.
- **`gpu`** uses `qwen3:8b`, a bigger sibling, for a GPU with at least 8 GB of **VRAM**: the GPU's own memory, separate from the computer's RAM.

Which fits *your* machine? `llmfit plan "Qwen/Qwen3-8B" --quant Q4_K_M --context 8192` tells you (guide `01b_llmfit`), and `lec_01b` explains what else goes into that choice.

</div>

<div style="text-align: justify; max-width: 90ch">

The cell also reads **`.env`**, your personal settings file (guide `01d_env_hugging_face`), with `load_dotenv`. An **environment variable** is a named value the operating system keeps for programs, outside your code;
`load_dotenv` copies each line of `.env` into one, and `os.environ.get("OLLAMA_HOST", ...)` reads it back, with a default for when `.env` does not set it.
`OLLAMA_HOST` is Ollama's address, which section 1 takes apart. To use another address in this notebook only, uncomment the `OLLAMA_HOST` or `MLFLOW_URI` line under `# To override .env`.
`MLFLOW_URI` and `EXPERIMENT` wait for lecture 2, where MLflow enters; they are here because the cell is the same in every notebook.

The last line calls a helper from `llm_course`, the small package installed by `uv sync`. It checks that Ollama answers and that the models are pulled, and if not it prints the exact command to run.

</div>

In [2]:
# --- Course configuration: edit TIER only; settings in .env (01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

import os
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # your settings file: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-basics"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. Two new tools

### `uv`: the same environment on every computer

A notebook that runs on one laptop can fail on another: a package is missing, or it has another version that behaves differently.
In the Python course you solved half of this with a **virtual environment**, a project's own folder of packages (`python -m venv`),
filled by `pip install -r requirements.txt`. The other half stayed open: a `requirements.txt` usually lists only the packages you asked for,
while the packages *they* need, their **dependencies**, and often the exact versions are whatever pip finds that day.

[**`uv`**](https://docs.astral.sh/uv/) does both jobs with one tool and closes the gap with a **lock file**, [`uv.lock`](https://docs.astral.sh/uv/concepts/projects/layout/#the-lockfile):
the exact version of every package, dependencies included. `uv sync` builds `.venv/` from it, so everyone runs the same code.
The guide `01a_git_uv` installs `uv` and maps each pip command to its `uv` equivalent.

</div>

<div style="text-align: justify; max-width: 90ch">

A notebook uses the interpreter you select as its kernel: in VS Code, choose the `teach-llm-system` environment the first time you run a notebook (guide `01a_git_uv`, section 6).

`pip install` still works, and that is the trap: it bypasses the lock file. In the comments below, **PATH** is the list of folders where the terminal looks for programs,
and `pyproject.toml` is the project's list of the packages it needs, the job `requirements.txt` did.

```bash
# WRONG: installs into whatever Python is first on PATH, invisible to uv.lock
pip install ollama

# RIGHT: adds the package to pyproject.toml and uv.lock, then installs it
uv add ollama
```

</div>

<div style="text-align: justify; max-width: 90ch">

### The Ollama application: a model server

A model is a large file of numbers, up to several gigabytes. To answer, something must load it into memory, which takes seconds,
keep it there, and run the computation on the CPU or the GPU. Loading it inside every notebook would repeat that wait and hold one copy per notebook.

[**Ollama**](https://docs.ollama.com/) solves this with a **server**: a program that keeps running in the background, holds the model in memory, and waits for requests.
Your code is the **client**: it sends a request and gets a response back. Ollama is a system-native application, installed like a browser, not like a Python package.
It runs **open models**: models whose **weights**, the numbers learned in training, anyone may download, such as the Qwen3 family this course uses.
`ollama pull <name>` downloads a model once into Ollama's own folder, outside the course.

</div>

<div style="text-align: justify; max-width: 90ch">

### What `http://localhost:11434` means

`OLLAMA_HOST` holds this address, and each part has a job:

| Part | Name | What it means |
| --- | --- | --- |
| `http://` | **HTTP** | the [protocol](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview) browsers use for web pages: the rules for sending a request and getting a response |
| `localhost` | **host** | which computer: every computer's name for itself (`127.0.0.1` in numbers), so nothing leaves the machine |
| `11434` | **port** | which server on that computer: each server listens on its own number; 11434 is Ollama's default |

`mlflow server` in lecture 13 of the Python course was the same idea, at `http://127.0.0.1:5000`; this course's MLflow listens on port 5010 (`MLFLOW_URI`), so the two never collide.

</div>

<div style="text-align: justify; max-width: 90ch">

An **API** (application programming interface) is the set of requests a server understands, each at its own path after the address ([Ollama's API](https://docs.ollama.com/api/introduction)).
Try two in a browser: <http://localhost:11434> answers `Ollama is running`, and <http://localhost:11434/api/tags> returns the pulled models as JSON,
a text format for data (`lec_01c`).

### The `ollama` Python package: a client

The `ollama` package runs no model. It is a **client library**: it turns a Python call into a request to the server, and the JSON reply into a Python object whose fields you read with a dot,
like `df.shape` (guide `01c_ollama` compares the two).

| Python call | Request it sends |
| --- | --- |
| `ollama.Client(host=OLLAMA_HOST)` | none; it stores the address |
| `client.list()` | `/api/tags`: the models on disk |
| `client.ps()` | `/api/ps`: the models loaded in memory |
| `client.chat(...)` | `/api/chat`: a conversation |

Let's ask the server what it has.

</div>

In [3]:
client = ollama.Client(host=OLLAMA_HOST)

models = client.list().models
pd.DataFrame(
    {
        "model": [m.model for m in models],
        "size_gb": [round(m.size / 1e9, 2) for m in models],
        "parameters": [m.details.parameter_size for m in models],
        "quantization": [m.details.quantization_level for m in models],
    }
)

,model,size_gb,parameters,quantization
0,qwen3:0.6b,0.52,751.63M,Q4_K_M
1,hf.co/unsloth/Qwen3-0.6B-GGUF:Q8_0,0.64,596M,unknown
2,hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0,0.64,596M,unknown
3,qwen3:1.7b,1.36,2.0B,Q4_K_M
4,gemma3:4b,3.34,4.3B,Q4_K_M
5,nomic-embed-text:latest,0.27,137M,F16
6,qwen3:8b,5.23,8.2B,Q4_K_M


<div style="text-align: justify; max-width: 90ch">

Your table lists the models you pulled; this one comes from a machine with a few more. Three columns deserve a first look:

- **parameters**: the number of weights. The coefficients of a scikit-learn `LinearRegression` are weights too, a handful of them.
  `qwen3:1.7b` shows `2.0B`: the model uses one table of 311 million weights twice, at its input and at its output, and its name counts it once while the file stores two copies.
- **quantization**: how many bits each weight is stored in. `Q4_K_M` stores most weights in 4 bits instead of the original 16,
  which is why 2 billion weights take 1.4 GB, not 4 GB. `lec_01b` returns to this.
- **size_gb**: what that costs on disk, and roughly in memory when the model is loaded.

[`nomic-embed-text`](https://ollama.com/library/nomic-embed-text) is not a chat model; lecture 2 uses it to turn text into numbers.

</div>

<div style="text-align: justify; max-width: 90ch">

### Local or hosted?

The alternative is a **hosted model**: a company runs it on its servers, and your client sends each request over the internet with an **API key**,
a secret that identifies your account. Running locally gives:

- **Privacy**: nothing you send leaves the machine, which matters for a company's documents.
- **No account, no bill per request**, and it works offline.
- **Stability**: a pulled model stays the same until you pull it again; a provider can change or retire a hosted one.

The price is **quality and speed**: the model must fit your memory, so it is smaller and makes more mistakes than the largest hosted ones,
and your hardware sets its speed. Hosted wins when the data may leave, the task needs the strongest model, or no machine fits.
`lec_01b` turns this into requirements; the optional `lec_01f` calls a hosted model.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Your first call

An LLM does one thing: given text, it predicts the next **token** (a word piece, section 3), appends it, and repeats until it emits a **stop token**,
a special token that means "done". Everything else, chat, coding, summarising, is that loop with a clever **prompt**, the text you send.

`client.chat` sends one request to `/api/chat`. Its arguments:

- **`model`**: which pulled model answers.
- **`messages`**: the conversation, a list of dicts, each with a `role` and `content`. For now there is one message, from you, the `user`;
  `lec_01c` adds the other roles.
- **`think=False`**: `qwen3:1.7b` is a **reasoning model**: by default it "thinks out loud" before answering, which is slow.
  This switches that off; `lec_01b` shows what thinking costs.
- **`options`**: generation settings ([full list](https://docs.ollama.com/modelfile#valid-parameters-and-values)), here `temperature` (section 5) and `num_predict`,
  the most tokens the answer may have.

</div>

In [4]:
response = client.chat(
    model=CHAT_MODEL,
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is data leakage in machine learning?",
        }
    ],
    think=False,
    options={
        "temperature": 0,
        "num_predict": 150,
    },  # num_predict caps the answer length
)
print(response.message.content)

Data leakage in machine learning refers to a situation where a model inadvertently learns relationships between the training data and the test data that are not intended or valid, leading to overly optimistic performance estimates.


<div style="text-align: justify; max-width: 90ch">

Read the answer as you would a classmate's: fluent and close, but muddled. Leakage means information from outside the training data, such as the test set, is used to build the model;
the answer speaks of the model learning "relationships between the training data and the test data". A fluent answer is not a correct one;
lecture 3 measures how often answers are right.

</div>

<div style="text-align: justify; max-width: 90ch">

The reply carries more than the text. `response` holds the fields of Ollama's [chat response](https://docs.ollama.com/api/chat):

| Field | What it holds |
| --- | --- |
| `message.content`, `message.role` | the answer, and who wrote it: `assistant` |
| `done_reason` | why generation stopped: `stop` (the model finished) or `length` (it hit `num_predict`) |
| `prompt_eval_count` | tokens the model read |
| `eval_count` | tokens it wrote |
| `eval_duration`, `total_duration` | time in nanoseconds (billionths of a second, hence `/ 1e9`); the total includes loading the model |

"eval" here means the model's computation, not an evaluation of quality.

</div>

In [ ]:
tokens_in = response.prompt_eval_count
tokens_out = response.eval_count
seconds_out = response.eval_duration / 1e9

print(f"prompt tokens : {tokens_in}")
print(f"answer tokens : {tokens_out}")
print(
    f"generation    : {seconds_out:.1f} s  ->  {tokens_out / seconds_out:.1f} tokens/s"
)
print(
    f"total         : {response.total_duration / 1e9:.1f} s (includes loading the model the first time)"
)

prompt tokens : 28
answer tokens : 38
generation    : 0.4 s  ->  108.3 tokens/s
total         : 1.5 s (includes loading the model the first time)


<div style="text-align: justify; max-width: 90ch">

The prompt counts 28 tokens for a 12-token question (section 3 shows how to count). Ollama added 16: markers for where a message starts and ends and who wrote it, plus a switch that turns thinking off.
They take time and space like any other token.

Tokens per second is the number to remember when you compare models or machines. In one example run on a laptop CPU, `qwen3:1.7b` writes 10-25 tokens/s;
on a GPU, over 100.

### ⏸ Pause point

Run `client.ps()` below and compare with your neighbours: is the model on the CPU or the GPU, and how much memory does it use?

</div>

In [5]:
loaded = client.ps().models
pd.DataFrame(
    {
        "model": [m.model for m in loaded],
        "memory_gb": [round(m.size / 1e9, 2) for m in loaded],
        "on_gpu_gb": [round(m.size_vram / 1e9, 2) for m in loaded],
    }
)

,model,memory_gb,on_gpu_gb
0,qwen3:1.7b,1.9,1.9


<div style="text-align: justify; max-width: 90ch">

`on_gpu_gb` equal to `memory_gb` means the whole model sits on a GPU; `0.0` means it runs on the CPU; a value in between means it did not fit and is split, which is slower.
In this run the model took 1.9 GB of memory for a 1.36 GB file: the rest holds the context window (section 4) and working space for the computation.

Ollama keeps a model loaded for five minutes after the last call ([FAQ](https://docs.ollama.com/faq)), so the second call is faster than the first.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Tokens

A model reads numbers, not letters. A **tokenizer** cuts text into **tokens**, each a word, a word piece, or a punctuation mark,
and replaces each with its **token ID**, its position in the **vocabulary**: the fixed list of every token the model knows, about 151 000 for Qwen3.
The vocabulary is fixed before training, so a tokenizer belongs to its model family. Tokens matter three times over:

- **Cost**: hosted models charge per token; local models spend time per token.
- **Capacity**: the context window (next section) is measured in tokens.
- **Speed**: generation is one token at a time.

</div>

<div style="text-align: justify; max-width: 90ch">

To count tokens without running the model, we load the tokenizer alone with [`transformers`](https://huggingface.co/docs/transformers/index),
Hugging Face's Python library for models. `AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")` downloads the tokenizer files (about 16 MB,
one-time) from the model's page on the **Hugging Face Hub**, the website where model makers publish their files ([`Qwen/Qwen3-1.7B`](https://huggingface.co/Qwen/Qwen3-1.7B);
`lec_01b` explores the Hub). It is the tokenizer that Ollama's `qwen3:1.7b` uses. `encode` turns text into token IDs; `convert_ids_to_tokens` shows the piece behind each ID.

</div>

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

english = "Data leakage happens when the test set influences training."
ids = tokenizer.encode(english)
print(len(english), "characters ->", len(ids), "tokens")
print(tokenizer.convert_ids_to_tokens(ids))

59 characters -> 10 tokens
['Data', 'Ġleakage', 'Ġhappens', 'Ġwhen', 'Ġthe', 'Ġtest', 'Ġset', 'Ġinfluences', 'Ġtraining', '.']


<div style="text-align: justify; max-width: 90ch">

`Ġ` marks a token that starts with a space: `Ġleakage` is " leakage". Every word of this sentence is a single token: 59 characters, 10 tokens, 5.9 characters per token.
A common rule of thumb for English text is about 4 characters per token; a sentence of common, whole words does better. Watch what happens to the same sentence in Greek.

</div>

In [ ]:
greek = "Διαρροή δεδομένων συμβαίνει όταν το test set επηρεάζει την εκπαίδευση."

rows = []
for label, text in [("English", english), ("Greek", greek)]:
    n_tokens = len(tokenizer.encode(text))
    rows.append(
        {
            "language": label,
            "characters": len(text),
            "tokens": n_tokens,
            "chars_per_token": round(len(text) / n_tokens, 1),
        }
    )
pd.DataFrame(rows)

,language,characters,tokens,chars_per_token
0,English,59,10,5.9
1,Greek,70,57,1.2


<div style="text-align: justify; max-width: 90ch">

Greek takes almost six times as many tokens (57 against 10) for the same sentence. A vocabulary is built from the frequent pieces of the training text,
where Greek is rare, so the tokenizer spells most Greek words letter by letter: `Διαρροή` alone is seven tokens. Two consequences you will meet again:

- **Capacity and speed**: a Greek prompt fills the context window almost six times faster, and a Greek answer takes almost six times longer to generate.
- **Model choice**: which model to pick for a Greek use case is partly a **tokenizer** question (`lec_01b`).

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. The context window

You can reread page 1 of a long report while reading page 40. A model cannot: it computes over a fixed number of tokens at once, and text outside that span does not exist for it.
The **context window** is that maximum number of tokens: your messages plus its answer.

You set the window per call with Ollama's option `num_ctx`. A bigger window costs memory, because Ollama reserves space for every token of it when it loads the model:
in one example run, `qwen3:1.7b` took 1.9 GB with a 4096-token window and 2.4 GB with 8192.

Let's put a fact at the *start* of a long text and ask about it. First we make sure the text is longer than the window we will use.

</div>

In [ ]:
needle = "The secret word for this lecture is PELICAN."
filler = " ".join(
    f"Note {i}: in the Python course we used pandas for tables, seaborn for plots, and scikit-learn for models."
    for i in range(60)
)
long_text = needle + " " + filler
question = "What is the secret word mentioned in the text? Answer with the word only."

print("tokens in the text:", len(tokenizer.encode(long_text)))

tokens in the text: 1622


In [ ]:
# WRONG: a 512-token window; the start of the text is cut off, the model never sees the needle
response_small = client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": long_text + "\n\n" + question}],
    think=False,
    options={"temperature": 0, "num_ctx": 512, "num_predict": 20},
)
print("num_ctx=512  ->", response_small.message.content.strip())

num_ctx=512  -> models


In [ ]:
# RIGHT: a window large enough for the whole text
response_large = client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": long_text + "\n\n" + question}],
    think=False,
    options={"temperature": 0, "num_ctx": 4096, "num_predict": 20},
)
print("num_ctx=4096 ->", response_large.message.content.strip())

num_ctx=4096 -> PELICAN


<div style="text-align: justify; max-width: 90ch">

With the 512-token window the model answered `models`, a word from the notes it could see, instead of `PELICAN`. The prompt did not fit, so Ollama **truncated** it:
it dropped the beginning of the text, needle included. Nothing warned you. The only trace is in the reply: `response_small.prompt_eval_count` is 512, the window, while the whole prompt has over 1 600 tokens.
Overflow is silent, which is why lecture 2 counts tokens before sending documents to a model.

</div>

<div style="text-align: justify; max-width: 90ch">

Each model also has a maximum window. `ollama show qwen3:1.7b` in a terminal prints it as `context length 40960`. Ollama's default is smaller, to save memory:
4096 tokens unless the GPU has 24 GB of VRAM or more ([context length](https://docs.ollama.com/context-length)). `ollama ps` shows the window of each loaded model in its `CONTEXT` column.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. Temperature

Ask a chat app the same question twice and you may get two different answers. Variety helps a brainstorm and hurts a test, so you need a way to control it.

At each step the model outputs a probability for *every* token in its vocabulary, and Ollama then **samples**: it draws the next token at random, weighted by those probabilities.
**Temperature** reshapes them before the draw: at `0` the most likely token is picked every time; higher values flatten the probabilities, so less likely tokens get their chance.

Without the option, Ollama uses the model's default, `0.6` for `qwen3:1.7b` (`ollama show` lists it under Parameters), so every call in this course sets it.
Same prompt, three runs at each setting.

</div>

In [ ]:
prompt = "Suggest a name for a coffee shop run by data scientists. Answer with the name only."

rows = []
for temperature in [0.0, 1.0]:
    for run in range(3):
        answer = client.chat(
            model=CHAT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            think=False,
            options={"temperature": temperature, "num_predict": 20},
        ).message.content.strip()
        rows.append({"temperature": temperature, "run": run + 1, "answer": answer})
pd.DataFrame(rows)

,temperature,run,answer
0,0.0,1,DataBrew
1,0.0,2,DataBrew
2,0.0,3,DataBrew
3,1.0,1,DataCafé
4,1.0,2,DataBrew
5,1.0,3,NeuroCafe


<div style="text-align: justify; max-width: 90ch">

At `0` the three runs agree; at `1` they differ, and your run may show other names than the table above.

### ⏸ Pause point

Which setting would you ship in a customer-facing product, and which in a brainstorming tool? In this course we use `temperature=0` almost everywhere, for the same reason you set `random_state=42` in scikit-learn:
when a number changes, we want to know it was our change, not luck. (Temperature 0 is *nearly* deterministic; on some hardware two runs can still differ.)

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- `uv sync` builds `.venv/` from the lock file `uv.lock`; `uv run` runs tools inside it; never `pip install`.
- Ollama is a **local model server** at `localhost:11434`; the `ollama` package is its client: `client.chat` asks a model, `client.ps()` shows where it runs.
- Local buys privacy, no bill and a stable model, at the cost of model size and speed.
- An LLM predicts the next **token**; the reply reports tokens in, tokens out, time, and why it stopped.
- The tokenizer decides the cost: Greek needs several times more tokens than English.
- The **context window** is a hard, silent limit measured in tokens; `num_ctx` sets it per call.
- **Temperature** trades reproducibility for variety; the course uses `0`.

**Next: `lec_01b`**, choosing a model. Size is only one of the things that decide which model you run; the requirements of the job decide the rest.

</div>